# Safe camera AVI → MP4 (with useful FFmpeg errors)

**Source is read-only.** The notebook only visits the specified camera folders, reads `.avi` videos and `.csv` timestamps, and writes new files under a separate output parent. It does **not** read analysis, imaging, or other experimental files.

Supports **both** known structures (choose one below):

- `BCI_rich_cams/mouse_0403L/20230703/cam/cam1/`
- `BMI_data/cage_0403/mouse_0403L/20230703/camera_data/exp/cam1/`

**Run the first conversion as a test (`MAX_VIDEOS = 1`).** If it succeeds, set `MAX_VIDEOS = None` and rerun the conversion cell for the rest. It skips existing outputs, including existing CSV copies.

If it fails, copy the **FFMPEG ERROR** printed by the notebook. No encoding settings should be changed without seeing that error.

In [2]:
from pathlib import Path
import os
import shutil
import subprocess

# EDIT THESE PATHS and LAYOUT; output must NOT be inside the source tree.
SOURCE_ROOT = Path(r"C:\Data\datasets\BCI_rich_cams")
OUTPUT_PARENT = Path(r"C:\Data\datasets\BCI_rich_cams_converted")
LAYOUT = "BCI_rich_cams"  # or "BMI_data" for cage/mouse/day/camera_data/exp/camN

MAX_VIDEOS = 1  # Test a single AVI first; change to None after a successful test.
CRF = 25        # Lower = higher visual quality and larger output.

SOURCE_ROOT = SOURCE_ROOT.expanduser().resolve()
OUTPUT_ROOT = (OUTPUT_PARENT.expanduser().resolve() / SOURCE_ROOT.name).resolve()


def is_within(path, root):
    return path == root or root in path.parents


def is_link(path):
    return path.is_symlink() or (hasattr(path, "is_junction") and path.is_junction())


def source_is_safe(path):
    if not is_within(path.resolve(), SOURCE_ROOT):
        return False
    part = path
    while part != SOURCE_ROOT:
        if is_link(part):
            return False
        part = part.parent
    return True


def check_output(path):
    # Recheck even for previously created output subfolders (including junctions).
    real_path = path.resolve()
    if is_within(real_path, SOURCE_ROOT):
        raise ValueError(f"UNSAFE: output would touch raw data: {path}")


if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(f"SOURCE_ROOT does not exist: {SOURCE_ROOT}")
if LAYOUT not in ("BCI_rich_cams", "BMI_data"):
    raise ValueError("LAYOUT must be 'BCI_rich_cams' or 'BMI_data'")
if is_within(OUTPUT_ROOT, SOURCE_ROOT) or is_within(SOURCE_ROOT, OUTPUT_ROOT):
    raise ValueError("UNSAFE: source and output trees overlap")
check_output(OUTPUT_ROOT)

ffmpeg = shutil.which("ffmpeg")
if ffmpeg is None:
    raise RuntimeError("ffmpeg not found in this notebook's environment")
print("FFmpeg executable:", ffmpeg)
print("Source (READ ONLY):", SOURCE_ROOT)
print("Output (NEW files only):", OUTPUT_ROOT)
print("Layout:", LAYOUT)
print("Test limit:", MAX_VIDEOS)

FFmpeg executable: C:\Users\alireza\miniconda3\envs\BCI_analysis\Library\bin\ffmpeg.EXE
Source (READ ONLY): C:\Data\datasets\BCI_rich_cams
Output (NEW files only): C:\Data\datasets\BCI_rich_cams_converted\BCI_rich_cams
Layout: BCI_rich_cams
Test limit: 1


In [3]:
# Find camera folders only; do not open unrelated data files.
if LAYOUT == "BCI_rich_cams":
    pattern = "mouse_*/*/cam/cam[1-4]"
else:
    pattern = "cage_*/mouse_*/*/camera_data/exp/cam[1-4]"

camera_dirs = sorted(
    p for p in SOURCE_ROOT.glob(pattern)
    if p.is_dir() and source_is_safe(p)
)
print("Camera folders found:", len(camera_dirs))
for p in camera_dirs[:5]:
    print("  ", p.relative_to(SOURCE_ROOT))
if len(camera_dirs) > 5:
    print("  ... and", len(camera_dirs) - 5, "more")
if not camera_dirs:
    raise RuntimeError("No camera folders found. Check SOURCE_ROOT and LAYOUT.")

Camera folders found: 4
   mouse_0403L\20230703\cam\cam1
   mouse_0403L\20230703\cam\cam2
   mouse_0403L\20230703\cam\cam3
   mouse_0403L\20230703\cam\cam4


In [4]:
# Convert AVI → MP4 and copy CSV. FFmpeg reads the original AVI; all output is separate.
# -n prevents FFmpeg from overwriting a preexisting output file.
# Print FFmpeg's return code and stderr if conversion fails.
converted = skipped_videos = copied_csv = skipped_csv = failed = attempted = 0

for cam_dir in camera_dirs:
    source_files = sorted(
        p for p in cam_dir.iterdir()
        if p.is_file() and source_is_safe(p)
        and p.suffix.lower() in (".avi", ".csv")
    )
    avi_files = [p for p in source_files if p.suffix.lower() == ".avi"]
    csv_files = [p for p in source_files if p.suffix.lower() == ".csv"]
    if not avi_files and not csv_files:
        continue

    dest_dir = OUTPUT_ROOT / cam_dir.relative_to(SOURCE_ROOT)
    check_output(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    check_output(dest_dir)

    for avi in avi_files:
        final = dest_dir / (avi.stem + ".mp4")
        partial = dest_dir / (avi.stem + ".partial.mp4")
        check_output(final)
        check_output(partial)
        if final.exists() or partial.exists():
            skipped_videos += 1
            print("SKIP video (already exists):", final)
            continue
        if MAX_VIDEOS is not None and attempted >= MAX_VIDEOS:
            break
        attempted += 1

        command = [
            ffmpeg, "-nostdin", "-hide_banner", "-loglevel", "info", "-n",
            "-i", str(avi), "-map", "0:v:0", "-map", "0:a?",
            "-c:v", "libx264", "-preset", "medium", "-crf", str(CRF),
            "-pix_fmt", "yuv420p", "-fps_mode", "passthrough",
            "-c:a", "aac", "-b:a", "128k", str(partial),
        ]
        print("\nCONVERT:", avi.relative_to(SOURCE_ROOT))
        try:
            result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            stderr = result.stderr.decode("utf-8", errors="replace")
            stdout = result.stdout.decode("utf-8", errors="replace")
            ok = result.returncode == 0 and partial.is_file() and partial.stat().st_size > 0
            if ok:
                # Publish without overwriting any pre-existing final video.
                # Hard-linking is atomic, no-clobber, and stays within OUTPUT only.
                try:
                    os.link(partial, final)
                except FileExistsError:
                    skipped_videos += 1
                    print("SKIP final (appeared concurrently):", final)
                except OSError:
                    # Network drives may not allow hard links: use exclusive creation.
                    try:
                        with partial.open("rb") as src, final.open("xb") as dst:
                            shutil.copyfileobj(src, dst)
                    except FileExistsError:
                        skipped_videos += 1
                        print("SKIP final (already exists):", final)
                    else:
                        converted += 1
                else:
                    converted += 1
                partial.unlink()  # ONLY temporary MP4 generated under OUTPUT_ROOT
                print("OK:", final)
            else:
                failed += 1
                print("FFMPEG FAILED -- return code:", result.returncode)
                print("Partial file exists:", partial.exists())
                print("FFMPEG STDERR (last 6000 characters):\n", stderr[-6000:] or "<empty>")
                if stdout.strip():
                    print("FFMPEG STDOUT (last 2000 characters):\n", stdout[-2000:])
                if partial.is_file():
                    partial.unlink()  # ONLY incomplete OUTPUT from this attempt
        except Exception as exc:
            failed += 1
            print("PYTHON ERROR:", type(exc).__name__, str(exc))
            # The exception is displayed rather than hidden; original AVI is not modified.

    # CSV files are copied to the same OUTPUT folder, never replacing an existing file.
    for csv in csv_files:
        target = dest_dir / csv.name
        check_output(target)
        try:
            with csv.open("rb") as src, target.open("xb") as dst:
                shutil.copyfileobj(src, dst)
            copied_csv += 1
        except FileExistsError:
            skipped_csv += 1
        # Other failures are intentionally not suppressed.

    if MAX_VIDEOS is not None and attempted >= MAX_VIDEOS:
        break

print(f"\nDone: converted={converted}; skipped AVI={skipped_videos}; "
      f"CSV copied={copied_csv}; CSV skipped={skipped_csv}; "
      f"conversion failures={failed}; attempts={attempted}")


CONVERT: mouse_0403L\20230703\cam\cam1\cam12023-07-25T11_19_05.avi
FFMPEG FAILED -- return code: 3221225785
Partial file exists: False
FFMPEG STDERR (last 6000 characters):
 <empty>

Done: converted=0; skipped AVI=0; CSV copied=0; CSV skipped=1; conversion failures=1; attempts=1
